# Init

In [220]:
init_training_sentences = ["the cat sat on the mat", "the dog sat on the chair", "the guy sat on the sofa"]
init_vocab_size = len(set([word for sentence in init_training_sentences for word in sentence.split()])) # unique words in training sentences
init_embedding_dimension = 8
init_attention_dimension = 4 # usually better if it's half of embedding dimension
init_input = "the cat sat"
init_input_length = len(init_input.split())

## Tokenization

In [221]:
# Get list of unique words

print("Tokenization\n")

unique_words = sorted(set([word for sentence in init_training_sentences for word in sentence.split()]))

print("unique_words: ", unique_words)

word_to_id = {
    word: i for i, word in enumerate(unique_words)
}

print("word_to_id: ", word_to_id)

id_to_word = {
    i: word for word, i in word_to_id.items()
}

# get input and encode it
input_encoded = [word_to_id[input] for input in init_input.split()]

print("\ninput_encoded: ", input_encoded)


Tokenization

unique_words:  ['cat', 'chair', 'dog', 'guy', 'mat', 'on', 'sat', 'sofa', 'the']
word_to_id:  {'cat': 0, 'chair': 1, 'dog': 2, 'guy': 3, 'mat': 4, 'on': 5, 'sat': 6, 'sofa': 7, 'the': 8}

input_encoded:  [8, 0, 6]


## Embedding

In [222]:
import numpy as np

print("Embedding\n")
embedding_table = np.random.randn(init_vocab_size, init_embedding_dimension)

print("embedding_table.shape: \n", embedding_table.shape)
print("input_encoded: \n", input_encoded)

embeddings = embedding_table[input_encoded]

print("\nembeddings.shape: ", embeddings.shape)

Embedding

embedding_table.shape: 
 (9, 8)
input_encoded: 
 [8, 0, 6]

embeddings.shape:  (3, 8)


## Attention

In [223]:
# Create attention layer

# Who am I?
W_K = np.random.randn(init_embedding_dimension, init_attention_dimension)
# What am I looking for?
W_Q = np.random.randn(init_embedding_dimension, init_attention_dimension)
# What information do I pass forward?
W_V = np.random.randn(init_embedding_dimension, init_attention_dimension)

print("embeddings.shape: ", embeddings.shape)
print("W_K.shape: ", W_K.shape)

# Create Keys
K = embeddings.dot(W_K)
# Create Queries
Q = embeddings.dot(W_Q)
# Create Values
V = embeddings.dot(W_V)

print("K.shape: embeddings @ W_K: ", K.shape)

# Compare every query against every key
scores = Q.dot(K.T)
print("Q: ", Q.shape)
print("Q @ K: ", scores.shape)

# Normalize scores
scores = scores / np.sqrt(init_attention_dimension)


embeddings.shape:  (3, 8)
W_K.shape:  (8, 4)
K.shape: embeddings @ W_K:  (3, 4)
Q:  (3, 4)
Q @ K:  (3, 3)


## Softmax

In [226]:
# Convert scores into probabilities
print("scores: \n\n", scores)

# upper triangle = future tokens
mask = np.triu(np.ones(scores.shape), k=1)

# Prevent future token attention
scores = np.where(mask == 1, -np.inf, scores)

print("causal masking:\n\n", scores)

scores = scores - np.max(scores, axis=-1, keepdims=True)

print("scores - max: \n\n", scores)

exp_scores = np.exp(scores)

print("exp_scores: \n\n", exp_scores)

attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

print("attention_weights: \n\n", attention_weights)

softmax = attention_weights.dot(V)

print("softmax: \n\n", softmax)

scores: 

 [[  0.                 -inf         -inf]
 [-20.47667913   0.                 -inf]
 [ -2.74584872   0.          -7.53428916]]
causal masking:

 [[  0.                 -inf         -inf]
 [-20.47667913   0.                 -inf]
 [ -2.74584872   0.          -7.53428916]]
scores - max: 

 [[  0.                 -inf         -inf]
 [-20.47667913   0.                 -inf]
 [ -2.74584872   0.          -7.53428916]]
exp_scores: 

 [[1.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.27965013e-09 1.00000000e+00 0.00000000e+00]
 [6.41937954e-02 1.00000000e+00 5.34441028e-04]]
attention_weights: 

 [[1.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.27965013e-09 9.99999999e-01 0.00000000e+00]
 [6.02912491e-02 9.39206800e-01 5.01950648e-04]]
softmax: 

 [[ 3.84906185 -1.89769918 -0.15539666 -0.51820263]
 [-1.01332863 -1.19460652  3.3280408  -6.57816704]
 [-0.7199538  -1.23624543  3.11895991 -6.20988614]]
